In [1]:
# Calculate and save OSDMA8

In [2]:
import os
import xarray as xr
from utils.utils import get_scenario_config

In [3]:
# === Processing function ===
def calculate_maximum_6month_mean(start_year, end_year, monthly_mda8):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")

    return annual_max_6m

In [5]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

MDA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/MDA8/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")

    dates = f"{years.start}0101-{years.stop}1231"
    # Final year will not be complete due to SH Jan-Mar missing
    new_dates = f"{years.start}-{years.stop - 1}"

    in_file = f"MDA8_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    file_list = [os.path.join(MDA8_DIR, in_file)]
    OSDMA8 = []

    for file in file_list:
        if not os.path.exists(file):
            print(f"Missing: {file}")
            continue

        print(f"Reading {os.path.basename(file)}")
        monthly_mda8 = xr.open_dataarray(file)

        # Create list of years to calculate over
        start_year = int(str(monthly_mda8.time.dt.year[0].values))
        # Take second to last year to keep final March
        end_year = int(str(monthly_mda8.time.dt.year[-1].values - 1))

        annual_max_6m = calculate_maximum_6month_mean(start_year, end_year,
                                                      monthly_mda8)

        OSDMA8.append(annual_max_6m)

    if OSDMA8:
        combined = xr.concat(OSDMA8, dim="year")
        # Remove the unused time dimension
        combined = combined.drop_vars("time")

        out_file = f"OSDMA8_{model}_{scenario}_{ens_num:02d}_{new_dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("OSDMA8: Highest seasonal (6-month) average of "
                       "8-hour daily maximum ozone concentrations across "
                       "15 months (Jan-Mar) - scripts by A.F. Wells (2025)")
        combined.attrs = monthly_mda8.attrs
        combined.attrs["description"] = description
        combined.attrs["ensemble_number"] = ens_num
        combined.attrs["scenario"] = scenario
        combined.attrs["model"] = model
        combined.to_netcdf(out_path)

print("All processing complete.")

Processing G6-1.5K, Ensemble 01
Reading MDA8_UKESM1_G6-1.5K_01_20350101-20841231.nc
Saving to /glade/work/awells/air_quality/UKESM1/ozone/OSDMA8/OSDMA8_UKESM1_G6-1.5K_01_2035-2083.nc
Processing G6-1.5K, Ensemble 02
Reading MDA8_UKESM1_G6-1.5K_02_20350101-20841231.nc
Saving to /glade/work/awells/air_quality/UKESM1/ozone/OSDMA8/OSDMA8_UKESM1_G6-1.5K_02_2035-2083.nc
Processing G6-1.5K, Ensemble 03
Reading MDA8_UKESM1_G6-1.5K_03_20350101-20841231.nc
Saving to /glade/work/awells/air_quality/UKESM1/ozone/OSDMA8/OSDMA8_UKESM1_G6-1.5K_03_2035-2083.nc
All processing complete.
